# Aula 6 - Imputação e Normalização de Dados

## Introdução

Contextualização

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
from sklearn.preprocessing import PowerTransformer

from google.colab import files

uploaded = files.upload()

imoveis = pd.read_csv("imoveis_sao_paulo.csv")

Saving imoveis_sao_paulo.csv to imoveis_sao_paulo (3).csv


## Preparação

Contextualização

In [2]:
imoveis = imoveis.rename(columns={
    "parking spaces": "parking_spaces",
    "hoa (R$)": "hoa",
    "rent amount (R$)": "rent_amount",
    "property tax (R$)": "property_tax",
    "total (R$)": "total",})

imoveis["floor"] = pd.to_numeric(imoveis["floor"], errors="coerce")
imoveis["property_type"] = np.where(imoveis["floor"].notna(), "apartment", "house")

print(imoveis["property_type"].value_counts(dropna=False))

property_type
apartment    25252
house         4454
Name: count, dtype: int64


## 1. Comparação entre o MAE da média e da mediana para `rooms`

Contextualização

In [3]:
rooms_obs = imoveis["rooms"].dropna()

rng = np.random.default_rng(42)
mascara = rng.random(len(rooms_obs)) < 0.20

treino = rooms_obs[~mascara]
teste = rooms_obs[mascara]

media_rooms = treino.mean()
mediana_rooms = treino.median()

mae_media = np.abs(teste - media_rooms).mean()
mae_mediana = np.abs(teste - mediana_rooms).mean()

print(f"Assimetria (skew): {rooms_obs.skew():.3f}")
print(f"Média: {media_rooms:.2f} | MAE: {mae_media:.4f}")
print(f"Mediana: {mediana_rooms:.2f} | MAE: {mae_mediana:.4f}")

Assimetria (skew): 0.374
Média: 2.26 | MAE: 1.0043
Mediana: 2.00 | MAE: 0.9472


Interpretação

## 2. Imputação de aluguel por tier e zona vs. bairro

Contextualização

In [4]:
mediana_tier_zona = imoveis.groupby(["tier", "zona"])["rent_amount"].transform("median")
rent_imp_tier_zona = (
    imoveis["rent_amount"]
    .fillna(mediana_tier_zona)
    .fillna(imoveis["rent_amount"].median()))

mediana_bairro = imoveis.groupby("bairro")["rent_amount"].transform("median")
rent_imp_bairro = (
    imoveis["rent_amount"]
    .fillna(mediana_bairro)
    .fillna(imoveis["rent_amount"].median()))

contagem_bairros = imoveis.groupby("bairro")["rent_amount"].count()
print("Resumo do tamanho dos grupos por bairro:")
print(contagem_bairros.describe())
print(f"\nBairros com menos de 5 observações válidas: {(contagem_bairros < 5).sum()}")
print(f"Bairros sem nenhuma observação válida: {(contagem_bairros == 0).sum()}")

Resumo do tamanho dos grupos por bairro:
count     85.000000
mean     349.482353
std       19.567716
min      313.000000
25%      334.000000
50%      351.000000
75%      366.000000
max      394.000000
Name: rent_amount, dtype: float64

Bairros com menos de 5 observações válidas: 0
Bairros sem nenhuma observação válida: 0


Interpretação

## 3. Comparação de transformações para hoa entre apartamentos

Contextualização

In [5]:
hoa_apt = imoveis.loc[
    (imoveis["property_type"] == "apartment") & (imoveis["hoa"] > 0),"hoa"]

hoa_log1p = np.log1p(hoa_apt)

pt_bc = PowerTransformer(method="box-cox", standardize=False)
hoa_bc = pt_bc.fit_transform(hoa_apt.to_frame()).ravel()

pt_yj = PowerTransformer(method="yeo-johnson", standardize=False)
hoa_yj = pt_yj.fit_transform(hoa_apt.to_frame()).ravel()

comparacao = pd.DataFrame({
    "Transformação": ["Original (hoa)", "log1p", "Box-Cox", "Yeo-Johnson"],
    "Lambda (λ)": [np.nan, 0.0, pt_bc.lambdas_[0], pt_yj.lambdas_[0]],
    "Assimetria (skew)": [
        hoa_apt.skew(),
        pd.Series(hoa_log1p).skew(),
        pd.Series(hoa_bc).skew(),
        pd.Series(hoa_yj).skew(),]})

print(comparacao.to_string(index=False))

 Transformação  Lambda (λ)  Assimetria (skew)
Original (hoa)         NaN           2.128776
         log1p    0.000000           0.149353
       Box-Cox   -0.085492           0.000406
   Yeo-Johnson   -0.086791           0.000435


Interpretação